In [2]:
pip install psycopg2-binary

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --------------- ------------------------ 1.0/2.8 MB 5.6 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.8 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 5.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: C:\Users\Antara\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [2]:
pip install python-dotenv


  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: C:\Users\Antara\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [3]:
import json
import psycopg2
import os
from dotenv import load_dotenv
from kafka import KafkaConsumer

In [20]:
BOOTSTRAP_SERVERS = ["localhost:9092"]
TOPIC = "stock_prices_v2"
GROUP_ID = "stock_price_writer"

In [17]:
PG_CONFIG = dict(
    host="localhost",
    port=5432,
    dbname="stockdb",
    user="stockuser",
    password="stockpass",
)

In [21]:
INSERT_SQL = """
      INSERT INTO stock_prices_v2 (ticker, price, fetched_at)
      VALUES (%(ticker)s, %(price)s, %(fetched_at)s)
"""

In [24]:
def build_consumer() -> KafkaConsumer:
    return KafkaConsumer(
        TOPIC,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=GROUP_ID,
        auto_offset_reset="earliest",
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    )

In [22]:
def main():
    consumer = build_consumer()
    conn = psycopg2.connect(**PG_CONFIG)
    conn.autocommit = True
    cur = conn.cursor()
    print(f"[consumer] listening on '{TOPIC}', writing into Postgres. Ctrl+C to stop.")
    try:
        for message in consumer:
            record = message.value
            cur.execute(INSERT_SQL, record)
            print(f"[consumer] wrote {record['ticker']} @ {record['price']}")
    except KeyboardInterrupt:
        print("\n[consume] stopped.")
    finally:
        cur.close()
        conn.close()
        consumer.close()

In [26]:
if __name__ == "__main__":
    main() 

C:\Users\Antara\AppData\Local\Temp\ipykernel_14168\4022541506.py:2: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  return KafkaConsumer(


[consumer] listening on 'stock_prices_v2', writing into Postgres. Ctrl+C to stop.
[consumer] wrote GOOGL @ 361.2250061035156
[consumer] wrote AAPL @ 312.20001220703125
[consumer] wrote MSFT @ 382.75
[consumer] wrote GOOGL @ 361.2250061035156
[consumer] wrote AAPL @ 312.17999267578125
[consumer] wrote MSFT @ 382.760009765625
[consumer] wrote GOOGL @ 361.1400146484375
[consumer] wrote AAPL @ 312.17999267578125
[consumer] wrote MSFT @ 382.760009765625
[consumer] wrote GOOGL @ 361.1400146484375
[consumer] wrote AAPL @ 312.7799987792969
[consumer] wrote MSFT @ 382.80999755859375
[consumer] wrote GOOGL @ 361.0799865722656
[consumer] wrote AAPL @ 312.7799987792969
[consumer] wrote MSFT @ 382.80999755859375
[consumer] wrote GOOGL @ 361.0799865722656
[consumer] wrote AAPL @ 312.93011474609375
[consumer] wrote MSFT @ 382.62200927734375
[consumer] wrote GOOGL @ 361.07000732421875
[consumer] wrote AAPL @ 312.93011474609375
[consumer] wrote MSFT @ 382.62200927734375
[consumer] wrote GOOGL @ 361.070

In [16]:
print(PG_CONFIG)

{'host': 'localhost', 'port': 5432, 'dbname': 'stockuser', 'password': 'stockpass'}
